# CareLoop Clinics — No-Show Guard
## Part 1: Python & Data Foundations (Ingest, Validate, Clean, Enrich)

**Project:** CareLoop Clinics: No-Show Guard — Data Ingestion & Enrichment  **Objective:** Build a robust, audited data engineering pipeline to clean raw booking exports, enrich clinic metadata with live weather telemetry, and generate verified database load scripts.

### Pipeline Overview:
1. **Task 1.1:** Low-level standard library CSV ingestion (`csv.DictReader`), type parsing, and rejection routing (`rejected_rows.json`).
2. **Task 1.2:** Pandas cleaning pass 1: structural audit, exact deduplication, booking channel standardization, and date/status coercion.
3. **Task 1.3:** Pandas cleaning pass 2: biological age validation, median imputation, and Tukey IQR outlier fencing.
4. **Task 1.4:** Feature engineering (`lead_days`, `no_show`) and vectorized NumPy summary calculations.
5. **Task 1.5:** External REST API integration (Open-Meteo) for weather enrichment (`clinics_enriched.csv`).
6. **Task 1.6:** Clean table export (`appointments_clean.csv`) and relational database SQL load script generation (`load_data.sql`).

### Environment Setup & Imports

In [1]:
import os
import csv
import json
import time
import requests
import numpy as np
import pandas as pd

# Ensure data directory exists
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

APPOINTMENTS_RAW = os.path.join(DATA_DIR, "appointments.csv")
CLINICS_RAW = os.path.join(DATA_DIR, "clinics.csv")
NOTES_RAW = os.path.join(DATA_DIR, "followup_notes.csv")

print("Environment initialized.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Environment initialized.
Pandas version: 2.2.3
NumPy version: 1.26.4


---
## Task 1.1: Read the file the foundations way

Using only the standard Python `csv` module (not pandas):
- Open `appointments.csv` with `newline=""` and read via `csv.DictReader`.
- Print the field header and total row count.
- Implement `parse_row(row)` converting `age`, `fee_inr`, `prior_no_shows` to `int` and `distance_km` to `float`.
- Route unparseable rows via `try/except ValueError` into `rejected_rows` and valid rows into `accepted_rows`.
- Dump rejected rows to `rejected_rows.json` (indent=2).

In [2]:
def parse_row(row):
    """
    Converts age, fee_inr, prior_no_shows to int and distance_km to float.
    Raises ValueError if conversion fails.
    """
    parsed = dict(row)
    parsed['age'] = int(row['age'])
    parsed['fee_inr'] = int(row['fee_inr'])
    parsed['prior_no_shows'] = int(row['prior_no_shows'])
    parsed['distance_km'] = float(row['distance_km'])
    return parsed

accepted_rows = []
rejected_rows = []

with open(APPOINTMENTS_RAW, mode="r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames
    print(f"Header columns: {headers}")
    
    total_raw_rows = 0
    for row in reader:
        total_raw_rows += 1
        try:
            parsed = parse_row(row)
            accepted_rows.append(parsed)
        except (ValueError, TypeError):
            rejected_rows.append(row)

print(f"Total rows read: {total_raw_rows}")
print(f"Accepted rows: {len(accepted_rows)}")
print(f"Rejected rows: {len(rejected_rows)}")

# Export rejected rows to JSON
with open("rejected_rows.json", "w", encoding="utf-8") as out_f:
    json.dump(rejected_rows, out_f, indent=2)

print("Task 1.1 Complete: rejected_rows.json successfully exported.")

Header columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows read: 1020
Accepted rows: 988
Rejected rows: 32
Task 1.1 Complete: rejected_rows.json successfully exported.


---
## Task 1.2: Inspect and clean, pass 1

Restarting afresh from raw `appointments.csv` using pandas:
- Check `shape`, `info()`, `isnull().sum()`, and `duplicated().sum()`.
- Sequence of cleaning operations:
  - (a) Drop exact duplicate rows.
  - (b) Standardize `booking_channel` with `.str.strip().str.title()`.
  - (c) Convert `appt_date` with `pd.to_datetime(..., errors='coerce')` and `booked_on` with `pd.to_datetime`.
  - (d) Drop rows where `appt_date` is missing after conversion or `status` is missing.
- Print the row count after each step.

In [3]:
# Load raw file afresh
df = pd.read_csv(APPOINTMENTS_RAW)

print("=== Initial Inspection ===")
print(f"Raw Shape: {df.shape}")
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows count: {df.duplicated().sum()}")

# Step (a): Drop exact duplicates
df = df.drop_duplicates()
print(f"Row count after dropping duplicates: {len(df)}")

# Step (b): Standardize booking_channel
print(f"Distinct raw booking_channel spellings before cleaning: {df['booking_channel'].nunique()}")
df['booking_channel'] = df['booking_channel'].astype(str).str.strip().str.title()
print(f"Distinct clean booking_channel categories: {df['booking_channel'].unique()}")

# Step (c): Parse dates
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'], errors='coerce')

# Step (d): Drop rows with missing appt_date or missing status
before_drop_d = len(df)
df = df.dropna(subset=['appt_date', 'status'])
df = df[df['status'].astype(str).str.strip() != ""]
print(f"Row count after step (d) (dropped unreadable dates / blank statuses): {len(df)}")

=== Initial Inspection ===
Raw Shape: (1020, 14)

Missing values per column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          1
age               12
gender             0
distance_km       20
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             4
dtype: int64

Duplicate rows count: 20
Row count after dropping duplicates: 1000
Distinct raw booking_channel spellings before cleaning: 16
Distinct clean booking_channel categories: ['App' 'Phone' 'Walk-In']
Row count after step (d) (dropped unreadable dates / blank statuses): 990


---
## Task 1.3: Clean, pass 2: validity, gaps and outliers

Order of operations:
- (a) Treat any `age < 1` or `age > 100` as invalid and set to missing (`np.nan`).
- (b) Fill missing `age` with median (`.round().astype(int)`) and missing `distance_km` with median.
- (c) Compute Q1, Q3, and IQR for `distance_km` and `fee_inr`. Cap values at $Q_3 + 1.5 \times IQR$ and $Q_1 - 1.5 \times IQR$ using `np.clip`.
- (d) Round `distance_km` to 1 decimal place and `fee_inr` to whole rupees (`.round().astype(int)`).
- Print the number of capped values in each column.

In [4]:
# (a) Treat age < 1 or age > 100 as invalid (missing)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
invalid_ages = (df['age'] < 1) | (df['age'] > 100)
print(f"Invalid biological ages identified: {invalid_ages.sum()}")
df.loc[invalid_ages, 'age'] = np.nan
print(f"Total missing ages to impute: {df['age'].isnull().sum()}")

# (b) Fill missing age and distance_km with medians
median_age = df['age'].median()
median_dist = df['distance_km'].median()
print(f"Median age: {median_age}, Median distance: {median_dist}")

df['age'] = df['age'].fillna(median_age).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(median_dist)

# (c) Tukey Outlier Fencing for distance_km and fee_inr
# Distance IQR
q1_dist = df['distance_km'].quantile(0.25)
q3_dist = df['distance_km'].quantile(0.75)
iqr_dist = q3_dist - q1_dist
upper_fence_dist = q3_dist + 1.5 * iqr_dist
lower_fence_dist = q1_dist - 1.5 * iqr_dist

capped_dist_count = ((df['distance_km'] > upper_fence_dist) | (df['distance_km'] < lower_fence_dist)).sum()
df['distance_km'] = np.clip(df['distance_km'], lower_fence_dist, upper_fence_dist)

# Fee IQR
q1_fee = df['fee_inr'].quantile(0.25)
q3_fee = df['fee_inr'].quantile(0.75)
iqr_fee = q3_fee - q1_fee
upper_fence_fee = q3_fee + 1.5 * iqr_fee
lower_fence_fee = q1_fee - 1.5 * iqr_fee

capped_fee_count = ((df['fee_inr'] > upper_fence_fee) | (df['fee_inr'] < lower_fence_fee)).sum()
df['fee_inr'] = np.clip(df['fee_inr'], lower_fence_fee, upper_fence_fee)

# (d) Rounding
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print(f"Distance Q1: {q1_dist}, Q3: {q3_dist}, Upper Fence: {upper_fence_dist}")
print(f"Values capped in distance_km: {capped_dist_count}")
print(f"Fee Q1: {q1_fee}, Q3: {q3_fee}, Upper Fence: {upper_fence_fee}")
print(f"Values capped in fee_inr: {capped_fee_count}")
print(f"Maximum fee post-capping: {df['fee_inr'].max()}")

Invalid biological ages identified: 5
Total missing ages to impute: 17
Median age: 48.0, Median distance: 8.25
Distance Q1: 4.625, Q3: 13.0, Upper Fence: 25.5625
Values capped in distance_km: 26
Fee Q1: 500.0, Q3: 787.5, Upper Fence: 1218.75
Values capped in fee_inr: 3
Maximum fee post-capping: 1219


---
## Task 1.4: Features and a NumPy summary

- Create `lead_days = (appt_date - booked_on).dt.days`.
- Create binary target: `no_show = 1 if status == 'No-Show' else 0`.
- Vectorized calculations via NumPy (no loops):
  - Column means of `['age', 'distance_km', 'fee_inr', 'lead_days']`.
  - Number and proportion of rows where `lead_days > 14`.
  - Population z-scores for `distance_km`: $(x - \mu) / \sigma$ and count of $|z| > 2$.

In [5]:
# Derive features
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)

print(f"Lead days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
print(f"Total no-shows: {df['no_show'].sum()} out of {len(df)} ({df['no_show'].mean() * 100:.1f}%)")

# NumPy vectorized summary
num_cols = ["age", "distance_km", "fee_inr", "lead_days"]
X_mat = df[num_cols].to_numpy()

# (a) Column means
col_means = np.mean(X_mat, axis=0)
print(f"Column means [age, distance_km, fee_inr, lead_days]:\n{np.round(col_means, 2)}")

# (b) Lead days > 14 boolean mask
lead_mask = df['lead_days'].to_numpy() > 14
print(f"Rows with lead_days > 14: {lead_mask.sum()} ({lead_mask.mean() * 100:.1f}%)")

# (c) Population z-scores for distance_km
dist_vals = df['distance_km'].to_numpy()
z_dist = (dist_vals - np.mean(dist_vals)) / np.std(dist_vals)
z_extreme_count = np.sum(np.abs(z_dist) > 2)
print(f"Count of rows with |z| > 2 for distance_km: {z_extreme_count}")

Lead days range: 1 to 45
Total no-shows: 218 out of 990 (22.0%)
Column means [age, distance_km, fee_inr, lead_days]:
[ 48.16   9.5  635.71  11.82]
Rows with lead_days > 14: 296 (29.9%)
Count of rows with |z| > 2 for distance_km: 53


---
## Task 1.5: Enrich the clinics from a live API

For each of the 8 clinics in `clinics.csv`:
- Query Open-Meteo REST API (`https://api.open-meteo.com/v1/forecast`) with latitude, longitude, and hourly temperature.
- Extract the midnight-UTC temperature (`hourly.temperature_2m[0]`) as `temp_c`.
- Fault tolerance: `timeout=10`, retry once on exception, assign `None` if unsuccessful.
- Save enriched table to `clinics_enriched.csv`.

In [6]:
clinics_df = pd.read_csv(CLINICS_RAW)
print(f"Loaded {len(clinics_df)} clinics:")
print(clinics_df[['clinic_id', 'clinic_name', 'city']])

temp_list = []
for idx, row in clinics_df.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    name = row['clinic_name']
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }
    
    temp_c = None
    for attempt in range(2):
        try:
            resp = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                if "hourly" in data and "temperature_2m" in data["hourly"]:
                    temp_c = data["hourly"]["temperature_2m"][0]
                    break
        except requests.exceptions.RequestException as e:
            if attempt == 1:
                print(f"Warning: Failed to fetch weather for {name} after retry: {e}")
    
    temp_list.append(temp_c)

clinics_df['temp_c'] = temp_list
clinics_df.to_csv("clinics_enriched.csv", index=False)
print("\nTask 1.5 Complete: clinics_enriched.csv successfully exported.")
print(clinics_df[['clinic_id', 'clinic_name', 'city', 'temp_c']])

Loaded 8 clinics:
  clinic_id                     clinic_name     city
0       C01            Hadapsar Care Centre     Pune
1       C02           Kothrud Family Clinic     Pune
2       C03            Malviya Nagar Clinic   Jaipur
3       C04           Vaishali Nagar Clinic   Jaipur
4       C05              Vijay Nagar Clinic   Indore
5       C06  Edappally Medical Centre Kochi    Kochi
6       C07          Gomti Nagar Care Point  Lucknow
7       C08          Sitabuldi Health Point   Nagpur

Task 1.5 Complete: clinics_enriched.csv successfully exported.
  clinic_id                     clinic_name     city  temp_c
0       C01            Hadapsar Care Centre     Pune    15.7
1       C02           Kothrud Family Clinic     Pune    14.4
2       C03            Malviya Nagar Clinic   Jaipur    15.3
3       C04           Vaishali Nagar Clinic   Jaipur    16.0
4       C05              Vijay Nagar Clinic   Indore    15.2
5       C06  Edappally Medical Centre Kochi    Kochi    24.2
6       C07   

---
## Task 1.6: Export the clean table and write the SQL load script

- Sort cleaned data by `appointment_id`.
- Format both `booked_on` and `appt_date` as `YYYY-MM-DD` text.
- Save `appointments_clean.csv` with exactly 16 designated columns in strict order.
- Generate `load_data.sql` with single-statement INSERTs (clinics first, then appointments).

In [7]:
# Verify sort and date formats
df = df.sort_values('appointment_id').reset_index(drop=True)
df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')
df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')

EXACT_COLS = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

df_clean = df[EXACT_COLS]

# Verification audits
assert df_clean['appointment_id'].nunique() == len(df_clean), "Appointment IDs must be strictly unique!"
assert df_clean.isnull().sum().sum() == 0, "Zero missing values allowed across all 16 columns!"

# Export clean dataset
df_clean.to_csv("appointments_clean.csv", index=False)
print(f"appointments_clean.csv exported with shape: {df_clean.shape}")

# Generate load_data.sql
with open("load_data.sql", "w", encoding="utf-8") as sql_f:
    sql_f.write("-- CareLoop Clinics: Automated Ingestion Script\n")
    sql_f.write("-- 1. Populate Clinics Master Table (Parent)\n")
    
    for _, c_row in clinics_df.iterrows():
        c_id = c_row['clinic_id']
        c_name = c_row['clinic_name']
        c_city = c_row['city']
        c_tier = f"'{c_row['tier']}'" if pd.notna(c_row['tier']) else "NULL"
        c_lat = c_row['latitude'] if pd.notna(c_row['latitude']) else "NULL"
        c_lon = c_row['longitude'] if pd.notna(c_row['longitude']) else "NULL"
        c_temp = c_row['temp_c'] if pd.notna(c_row['temp_c']) else "NULL"
        
        sql_f.write(f"INSERT INTO clinics VALUES ('{c_id}', '{c_name}', '{c_city}', {c_tier}, {c_lat}, {c_lon}, {c_temp});\n")
        
    sql_f.write("\n-- 2. Populate Appointments Table (Child)\n")
    for _, ar in df_clean.iterrows():
        stmt = (
            f"INSERT INTO appointments VALUES ("
            f"'{ar['appointment_id']}', '{ar['patient_id']}', '{ar['clinic_id']}', "
            f"'{ar['booked_on']}', '{ar['appt_date']}', {int(ar['age'])}, "
            f"'{ar['gender']}', {float(ar['distance_km'])}, {int(ar['fee_inr'])}, "
            f"'{ar['booking_channel']}', '{ar['visit_type']}', '{ar['reminder_sent']}', "
            f"{int(ar['prior_no_shows'])}, '{ar['status']}', {int(ar['lead_days'])}, "
            f"{int(ar['no_show'])});
"
        )
        sql_f.write(stmt)

print("Task 1.6 Complete: load_data.sql successfully generated with 998 INSERT statements.")

appointments_clean.csv exported with shape: (990, 16)
Task 1.6 Complete: load_data.sql successfully generated with 998 INSERT statements.
